# INV-M365-CA — Authoritative Persona Department Pack Rebase v1

Purpose: prove the H4S department-pack rebase from the staged H3 authoritative registry.

Lemma mapping: `L79`.

Invariant support: `invariants/lemmas/L79_m365_authoritative_persona_department_pack_rebase_v1.yaml`.

Expected results: seven preflight mismatches are recorded, all ten department packs reconcile to the staged H3 counts, no contract-only persona declares actions, and deterministic replay hashes match.

In [1]:

from __future__ import annotations

import hashlib
import json
from pathlib import Path

import yaml

repo = Path.cwd()
registry = yaml.safe_load((repo / 'registry' / 'persona_registry_v2.yaml').read_text())['personas']
pre_h4s_authority_counts = {
    'communication': {'total': 1, 'active': 1, 'registry_backed': 1},
    'design': {'total': 5, 'active': 5, 'registry_backed': 5},
    'engineering': {'total': 7, 'active': 7, 'registry_backed': 7},
    'hr': {'total': 1, 'active': 1, 'registry_backed': 1},
    'marketing': {'total': 7, 'active': 2, 'registry_backed': 2},
    'operations': {'total': 2, 'active': 2, 'registry_backed': 2},
    'product': {'total': 3, 'active': 3, 'registry_backed': 3},
    'project-management': {'total': 3, 'active': 3, 'registry_backed': 3},
    'studio-operations': {'total': 5, 'active': 5, 'registry_backed': 5},
    'testing': {'total': 5, 'active': 5, 'registry_backed': 5},
}
departments = list(pre_h4s_authority_counts)

def registry_counts(department: str) -> dict[str, int]:
    ids = [pid for pid, payload in registry.items() if payload['department'] == department]
    return {
        'total': len(ids),
        'active': sum(1 for pid in ids if registry[pid]['status'] == 'active'),
        'registry_backed': sum(1 for pid in ids if registry[pid]['coverage_status'] == 'registry-backed'),
        'supported_actions': sum(len(registry[pid].get('allowed_actions') or []) for pid in ids),
    }

current_registry_counts = {department: registry_counts(department) for department in departments}
preflight_mismatches = {
    department: {
        'pre_h4s_authority': pre_h4s_authority_counts[department],
        'staged_registry': {
            'total': current_registry_counts[department]['total'],
            'active': current_registry_counts[department]['active'],
            'registry_backed': current_registry_counts[department]['registry_backed'],
        },
    }
    for department in departments
    if pre_h4s_authority_counts[department] != {
        'total': current_registry_counts[department]['total'],
        'active': current_registry_counts[department]['active'],
        'registry_backed': current_registry_counts[department]['registry_backed'],
    }
}
assert sorted(preflight_mismatches) == [
    'communication',
    'engineering',
    'hr',
    'marketing',
    'operations',
    'project-management',
    'studio-operations',
]
preflight_mismatches


{'communication': {'pre_h4s_authority': {'total': 1,
   'active': 1,
   'registry_backed': 1},
  'staged_registry': {'total': 4, 'active': 1, 'registry_backed': 1}},
 'engineering': {'pre_h4s_authority': {'total': 7,
   'active': 7,
   'registry_backed': 7},
  'staged_registry': {'total': 8, 'active': 7, 'registry_backed': 7}},
 'hr': {'pre_h4s_authority': {'total': 1, 'active': 1, 'registry_backed': 1},
  'staged_registry': {'total': 2, 'active': 1, 'registry_backed': 1}},
 'marketing': {'pre_h4s_authority': {'total': 7,
   'active': 2,
   'registry_backed': 2},
  'staged_registry': {'total': 8, 'active': 2, 'registry_backed': 2}},
 'operations': {'pre_h4s_authority': {'total': 2,
   'active': 2,
   'registry_backed': 2},
  'staged_registry': {'total': 10, 'active': 2, 'registry_backed': 2}},
 'project-management': {'pre_h4s_authority': {'total': 3,
   'active': 3,
   'registry_backed': 3},
  'staged_registry': {'total': 5, 'active': 3, 'registry_backed': 3}},
 'studio-operations': {'

Reconcile the current department-pack authorities against the staged H3 registry truth and fail if any stale count remains.

In [2]:

def authority_counts(department: str) -> dict[str, int]:
    pack = yaml.safe_load((repo / 'registry' / f"department_pack_{department.replace('-', '_')}_v1.yaml").read_text())
    personas = pack['personas']
    registry_backed = sum(1 for persona in personas.values() if persona['coverage_status'] == 'registry-backed')
    active = sum(1 for persona_id in personas if registry[persona_id]['status'] == 'active')
    contract_only_action_violations = [
        persona_id
        for persona_id, persona in personas.items()
        if persona['coverage_status'] == 'persona-contract-only' and persona['supported_actions']
    ]
    if contract_only_action_violations:
        raise AssertionError(f"contract_only_actions_declared:{department}:{contract_only_action_violations}")
    return {
        'required_personas': int(pack['kpis']['required_personas']),
        'required_active_personas': int(pack['kpis']['required_active_personas']),
        'required_registry_backed_personas': int(pack['kpis']['required_registry_backed_personas']),
        'supported_action_count': int(pack['kpis']['supported_action_count']),
        'persona_count': len(personas),
        'active_count': active,
        'registry_backed_count': registry_backed,
    }

post_h4s_departments = {}
for department in departments:
    authority = authority_counts(department)
    registry_count = current_registry_counts[department]
    assert authority['required_personas'] == registry_count['total']
    assert authority['required_active_personas'] == registry_count['active']
    assert authority['required_registry_backed_personas'] == registry_count['registry_backed']
    assert authority['supported_action_count'] == registry_count['supported_actions']
    assert authority['persona_count'] == registry_count['total']
    assert authority['active_count'] == registry_count['active']
    assert authority['registry_backed_count'] == registry_count['registry_backed']
    post_h4s_departments[department] = {
        'registry': registry_count,
        'authority': authority,
    }
post_h4s_departments


{'communication': {'registry': {'total': 4,
   'active': 1,
   'registry_backed': 1,
   'supported_actions': 7},
  'authority': {'required_personas': 4,
   'required_active_personas': 1,
   'required_registry_backed_personas': 1,
   'supported_action_count': 7,
   'persona_count': 4,
   'active_count': 1,
   'registry_backed_count': 1}},
 'design': {'registry': {'total': 5,
   'active': 5,
   'registry_backed': 5,
   'supported_actions': 36},
  'authority': {'required_personas': 5,
   'required_active_personas': 5,
   'required_registry_backed_personas': 5,
   'supported_action_count': 36,
   'persona_count': 5,
   'active_count': 5,
   'registry_backed_count': 5}},
 'engineering': {'registry': {'total': 8,
   'active': 7,
   'registry_backed': 7,
   'supported_actions': 62},
  'authority': {'required_personas': 8,
   'required_active_personas': 7,
   'required_registry_backed_personas': 7,
   'supported_action_count': 62,
   'persona_count': 8,
   'active_count': 7,
   'registry_backe

Write the machine-readable H4S proof artifact and certify deterministic replay by hashing the same reconciliation payload twice.

In [3]:
payload = {
    'lemma_id': 'L79',
    'preflight_mismatch_departments': sorted(preflight_mismatches),
    'preflight_mismatch_count': len(preflight_mismatches),
    'post_h4s_departments': post_h4s_departments,
    'determinism_inputs': {
        'registry': 'registry/persona_registry_v2.yaml',
        'pack_glob': 'registry/department_pack_*_v1.yaml',
    },
}
serialized = json.dumps(payload, indent=2, sort_keys=True)
hash_one = hashlib.sha256(serialized.encode('utf-8')).hexdigest()
hash_two = hashlib.sha256(json.dumps(payload, indent=2, sort_keys=True).encode('utf-8')).hexdigest()
assert hash_one == hash_two
payload['deterministic_hash'] = hash_one
output_path = repo / 'configs' / 'generated' / 'authoritative_persona_department_pack_rebase_v1_verification.json'
output_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')
{'output_path': str(output_path), 'deterministic_hash': hash_one}


{'output_path': '/Users/smarthaus/Projects/GitHub/M365/configs/generated/authoritative_persona_department_pack_rebase_v1_verification.json',
 'deterministic_hash': '4f6933dd56f180b5c58eca28c25d25cae8b578deb2061cb8ee6ac99dcf806cbd'}